In [1]:
import requests
import pandas as pd

tabela   = 1737        # IPCA – série histórica
variavel = 63          # IPCA – variação mensal
periodo  = 209901   # últimos 12 meses


In [2]:
def buscar_sidra(tabela, variavel, periodo, nivel="n1"):
    """Busca uma série do SIDRA/IBGE e devolve um DataFrame."""
    url = f"https://apisidra.ibge.gov.br/values/t/{tabela}/{nivel}/all/v/{variavel}/p/{periodo}"
    
    resposta = requests.get(url, timeout=30)
    
    
    if resposta.status_code != 200:
        raise RuntimeError(f"Erro {resposta.status_code}: {resposta.text}")
    
    dados = resposta.json()
    
    if len(dados) <= 1:
        print("Sem dados para o período")
        return pd.DataFrame()
    
    cabecalho = dados[0]
    linhas = dados[1:]
    df = pd.DataFrame(linhas).rename(columns=cabecalho)
    return df

In [3]:
df_mensal = buscar_sidra(7060,63,"last 12","n7")

print(df_mensal.shape)
df_mensal.head()


(120, 13)


,Nível Territorial (Código),Nível Territorial,Unidade de Medida (Código),Unidade de Medida,Valor,Região Metropolitana até 2020 (Código),Região Metropolitana até 2020,Variável (Código),Variável,Mês (Código),Mês,"Geral, grupo, subgrupo, item e subitem (Código)","Geral, grupo, subgrupo, item e subitem"
0,7,Região Metropolitana até 2020,2,%,0.26,1501,Belém - PA,63,IPCA - Variação mensal,202510,outubro 2025,7169,Índice geral
1,7,Região Metropolitana até 2020,2,%,0.11,1501,Belém - PA,63,IPCA - Variação mensal,202511,novembro 2025,7169,Índice geral
2,7,Região Metropolitana até 2020,2,%,-0.10,1501,Belém - PA,63,IPCA - Variação mensal,202512,dezembro 2025,7169,Índice geral
3,7,Região Metropolitana até 2020,2,%,0.16,1501,Belém - PA,63,IPCA - Variação mensal,202601,janeiro 2026,7169,Índice geral
4,7,Região Metropolitana até 2020,2,%,0.62,1501,Belém - PA,63,IPCA - Variação mensal,202602,fevereiro 2026,7169,Índice geral


In [4]:
def transformar_sidra(df):
    """Converte tipos, seleciona e renomeia as colunas do SIDRA"""
    if df.empty:
        return df

    df = df.copy()
    
    #Converter valor e Mês (Código)
    df["Valor"] = pd.to_numeric(df["Valor"], errors ="coerce")
    df["Mês (Código)"] = pd.to_datetime(df["Mês (Código)"], format="%Y%m")
    #Selecionar e renomear colunas#
    df = df[[
        "Nível Territorial (Código)",
        "Unidade de Medida (Código)",
        "Valor",
        "Região Metropolitana até 2020 (Código)",
        "Região Metropolitana até 2020",
        "Variável (Código)",
        "Variável",
        "Mês (Código)",
        "Geral, grupo, subgrupo, item e subitem (Código)",
        "Geral, grupo, subgrupo, item e subitem"
    ]].rename(columns={
        "Nível Territorial (Código)": "nivel_id",
        "Unidade de Medida (Código)": "unidade_medida_id",
        "Valor": "val",
        "Região Metropolitana até 2020 (Código)": "regiao_2020_id",
        "Região Metropolitana até 2020": "regiao_2020",
        "Variável (Código)": "variavel_id",
        "Variável": "variavel",
        "Mês (Código)": "data",
        "Geral, grupo, subgrupo, item e subitem (Código)": "item_id",
        "Geral, grupo, subgrupo, item e subitem": "item"
    })
    return df

In [5]:
df_cru   = buscar_sidra(7060, 63, "last 12", nivel="n7")
df_ipca  = transformar_sidra(df_cru)

print(df_ipca.dtypes)
df_ipca.head()

nivel_id                     object
unidade_medida_id            object
val                         float64
regiao_2020_id               object
regiao_2020                  object
variavel_id                  object
variavel                     object
data                 datetime64[ns]
item_id                      object
item                         object
dtype: object


,nivel_id,unidade_medida_id,val,regiao_2020_id,regiao_2020,variavel_id,variavel,data,item_id,item
0,7,2,0.26,1501,Belém - PA,63,IPCA - Variação mensal,2025-10-01,7169,Índice geral
1,7,2,0.11,1501,Belém - PA,63,IPCA - Variação mensal,2025-11-01,7169,Índice geral
2,7,2,-0.10,1501,Belém - PA,63,IPCA - Variação mensal,2025-12-01,7169,Índice geral
3,7,2,0.16,1501,Belém - PA,63,IPCA - Variação mensal,2026-01-01,7169,Índice geral
4,7,2,0.62,1501,Belém - PA,63,IPCA - Variação mensal,2026-02-01,7169,Índice geral


In [6]:
df_teste = transformar_sidra(pd.DataFrame())
df_teste

""


In [12]:
def carregar(df, camada, nome):
    if df.empty:
        print(f"{nome}: DataFrame vazio, nada gravado")
        return
    
    pasta = Path("dados")/camada
    pasta.mkdir(parents=True, exist_ok=True)
    
    arquivo = pasta/f"{nome}.parquet"
    
    df.to_parquet(arquivo, index=False)
    
    print(f"{len(df)} linhas. Linhas gravadas em {arquivo}")

In [13]:
from pathlib import Path
df_cru  = buscar_sidra(7060, 63, "last 12", nivel="n7")
df_ipca = transformar_sidra(df_cru)

carregar(df_cru,  "bronze", "ipca_regiao")
carregar(df_ipca, "silver", "ipca_regiao")

120 linhas. Linhas gravadas em dados\bronze\ipca_regiao.parquet
120 linhas. Linhas gravadas em dados\silver\ipca_regiao.parquet


In [18]:
pd.read_parquet("dados/bronze/ipca_regiao.parquet").dtypes

Nível Territorial (Código)                         object
Nível Territorial                                  object
Unidade de Medida (Código)                         object
Unidade de Medida                                  object
Valor                                              object
Região Metropolitana até 2020 (Código)             object
Região Metropolitana até 2020                      object
Variável (Código)                                  object
Variável                                           object
Mês (Código)                                       object
Mês                                                object
Geral, grupo, subgrupo, item e subitem (Código)    object
Geral, grupo, subgrupo, item e subitem             object
dtype: object

In [19]:
pd.read_parquet("dados/silver/ipca_regiao.parquet").dtypes

nivel_id                     object
unidade_medida_id            object
val                         float64
regiao_2020_id               object
regiao_2020                  object
variavel_id                  object
variavel                     object
data                 datetime64[ns]
item_id                      object
item                         object
dtype: object